# Image-to-3D Benchmark — BATCH 2: SPAR3D + TRELLIS
Evaluates **SPAR3D** and **TRELLIS** on the curated GSO benchmark.

In [ ]:
# --- Error logging setup ---
import sys, traceback, datetime
from pathlib import Path
LOG_FILE = Path("/kaggle/working/outputs/benchmark_log.txt")
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)
class Tee:
    def __init__(self, *files):
        self.files = files
    def write(self, data):
        for f in self.files:
            f.write(data)
            f.flush()
    def flush(self):
        for f in self.files:
            f.flush()
log_f = open(LOG_FILE, "a")
sys.stdout = Tee(sys.stdout, log_f)
sys.stderr = Tee(sys.stderr, log_f)
print(f"=== BENCHMARK LOG START {datetime.datetime.now()} ===")

In [ ]:
import os, subprocess, sys, glob, gc, json
from pathlib import Path

# 1. Headless OpenGL & T4 CUDA compilation flags (sm_75 only -> 10x faster C++/CUDA builds!)
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["ATTN_BACKEND"] = "xformers"
os.environ["SPCONV_ALGO"] = "native"
os.environ["SPCONV_DISABLE_JIT"] = "1"
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
os.environ["MAX_JOBS"] = "4"

# 2. HuggingFace token check
_hf_token = None
try:
    for _k in ["HF_TOKEN", "hf_token", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"]:
        if os.environ.get(_k):
            _hf_token = os.environ[_k]
            print(f"HF_TOKEN found in environment via {_k} ✅")
            break
    if not _hf_token:
        from kaggle_secrets import UserSecretsClient
        _client = UserSecretsClient()
        for _k in ["HF_TOKEN", "hf_token", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"]:
            try:
                _val = _client.get_secret(_k)
                if _val:
                    _hf_token = _val
                    print(f"HF_TOKEN found in Kaggle Secrets under '{_k}' ✅")
                    break
            except Exception:
                pass
except Exception as _e:
    print(f"Notice: Kaggle secrets check: {type(_e).__name__}: {_e}")

if _hf_token:
    os.environ["HF_TOKEN"] = _hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = _hf_token
    os.environ["HUGGINGFACE_TOKEN"] = _hf_token
    try:
        from huggingface_hub import login
        login(token=_hf_token, write_to_disk=True)
    except Exception:
        pass
    print("HF_TOKEN configured ✅")
else:
    print("Notice: HF_TOKEN not attached (public/mirror models will still work).")

import torch
print(f"torch {torch.__version__} (CUDA: {torch.version.cuda})")
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU attached! Enable Accelerator > GPU T4 x2.")
props = torch.cuda.get_device_properties(0)
print(f"CUDA OK: {props.name} | {props.total_memory/1e9:.1f} GB VRAM")

NUM_SAMPLES = 100
GSO_DIR = "/kaggle/working/data/gso"
OUT = Path("/kaggle/working/outputs"); OUT.mkdir(parents=True, exist_ok=True)

import numpy
open("/tmp/constraints.txt", "w").write(f"numpy=={numpy.__version__}\n")
print(f"Config: NUM_SAMPLES={NUM_SAMPLES} | numpy pinned: {numpy.__version__}")

In [ ]:
import os, sys, subprocess, datetime
import numpy as _np
if not os.path.exists("/tmp/constraints.txt"):
    open("/tmp/constraints.txt", "w").write(f"numpy=={_np.__version__}\n")

# Install pure wheels respecting Kaggle numpy==2.1.3 constraint, then install
# lpips/rembg/pyrender with --no-deps so rembg>=2.0.85 (which pins numpy>=2.3.0)
# never forces pip to backtrack scikit-image/scipy to a 22.7 MB source tarball.
!pip install -q -c /tmp/constraints.txt trimesh onnxruntime pooch pymatting freetype-py pyglet "PyOpenGL>=3.1.7"
!pip install -q --no-deps lpips rembg pyrender

import torch
assert torch.cuda.is_available(), "torch lost CUDA during installs"
import lpips
_lpips_vgg = lpips.LPIPS(net="vgg")
del _lpips_vgg
from rembg import new_session
_sess = new_session("isnet-general-use")
print("=== CELL 3 GATE: PASS (LPIPS + ISNet cached) ✅ ===")

In [ ]:
import os, sys, shutil, hashlib, subprocess

CODE_DIR = "/kaggle/working/image-to-3d-benchmark"
shutil.rmtree(CODE_DIR, ignore_errors=True)
_r = subprocess.run(
    ["git", "clone", "--depth", "1", "-b", "main", "https://github.com/anubhavaanand/image-to-3d.git", CODE_DIR],
    capture_output=True, text=True
)
if _r.returncode != 0 or not os.path.exists(os.path.join(CODE_DIR, "src")):
    raise RuntimeError("GitHub clone failed:\n" + _r.stderr[-500:])

_sha = subprocess.run(["git", "-C", CODE_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

for _k in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    del sys.modules[_k]
from src.evaluation.benchmark_runner import BenchmarkRunner, load_gso_samples, bake_texture_to_vertices
from src.engines.triposr_engine import TriposrEngine
from src.engines.lgm_engine import LGMEngine
from src.engines.trellis_engine import TrellisEngine
from src.engines.spar3d_engine import Spar3dEngine
os.makedirs("/kaggle/working/outputs", exist_ok=True)
with open("/kaggle/working/outputs/code_sha.txt", "w") as _f:
    _f.write(_sha + "\n")
print(f"code cloned fresh @ {_sha[:12]} — all 4 engines imported OK ✅ (sha recorded in outputs/code_sha.txt)")

In [ ]:
import os, sys, glob, zipfile, tempfile, shutil, random, time as _time
import json
from pathlib import Path
import numpy as np, trimesh
from PIL import Image as PILImage

os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
if "GSO_DIR" not in globals():
    GSO_DIR = "/kaggle/working/data/gso"
if "NUM_SAMPLES" not in globals():
    NUM_SAMPLES = 100
if "CODE_DIR" not in globals():
    CODE_DIR = "/kaggle/working/image-to-3d-benchmark"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src.evaluation.benchmark_runner import load_gso_samples, bake_texture_to_vertices, render_gso_mesh_view

CANONICAL_10_GSO = [    "ASICS_GELLinksmaster_WhiteCoffeeSand",
    "3D_Dollhouse_Happy_Brother",
    "3D_Dollhouse_Sofa",
    "Bradshaw_International_11642_7_Qt_MP_Plastic_Bowl",
    "Android_Figure_Panda",
    "Animal_Planet_Foam_2Headed_Dragon",
    "Black_and_Decker_TR3500SD_2Slice_Toaster",
    "Down_To_Earth_Orchid_Pot_Ceramic_Red",
    "Dog",
    "Dino_3",
]
if NUM_SAMPLES <= 10:
    TARGET_OBJECTS = CANONICAL_10_GSO[:NUM_SAMPLES]
else:
    _hp_list = []
    for _hp in ["/kaggle/working/handpicked.txt", os.path.join(CODE_DIR, "handpicked.txt")] + glob.glob("/kaggle/input/*/handpicked.txt"):
        if os.path.exists(_hp):
            _hp_list = [l.strip() for l in open(_hp) if l.strip() and not l.strip().startswith("#")]
            break
    _seen = set()
    TARGET_OBJECTS = []
    for _o in CANONICAL_10_GSO + _hp_list:
        if _o not in _seen:
            _seen.add(_o)
            TARGET_OBJECTS.append(_o)
        if len(TARGET_OBJECTS) >= NUM_SAMPLES:
            break
assert len(TARGET_OBJECTS) == NUM_SAMPLES, f"TARGET_OBJECTS has {len(TARGET_OBJECTS)} but NUM_SAMPLES={NUM_SAMPLES}"
assert len(set(TARGET_OBJECTS)) == len(TARGET_OBJECTS), "duplicate object in TARGET_OBJECTS"

def count_valid(root):
    root = Path(root)
    if not root.exists():
        return 0
    return sum(
        1 for d in root.iterdir()
        if d.is_dir() and (list((d/"meshes").glob("*.obj")) or list((d/"meshes").glob("*.ply")))
    )

REPO = "suvadityamuk/google-scanned-objects-raw"
def have_all(root):
    root = Path(root)
    for n in TARGET_OBJECTS:
        m = root / n / "meshes"
        if not (m.is_dir() and (list(m.glob("*.obj")) or list(m.glob("*.ply")))):
            return False
    return True
if not have_all(GSO_DIR):
    from huggingface_hub import HfApi, hf_hub_download
    api = HfApi(token=os.environ.get("HF_TOKEN"))
    all_zips = sorted(f for f in api.list_repo_files(REPO, repo_type="dataset") if f.endswith(".zip"))
    _zset = set(all_zips)
    _missing = [n for n in TARGET_OBJECTS if (n + ".zip") not in _zset]
    if _missing:
        raise RuntimeError(f"GSO zips missing upstream: {_missing}")
    wanted = [n + ".zip" for n in TARGET_OBJECTS]

    def _has_mesh(obj_id):
        m = Path(GSO_DIR) / obj_id / "meshes"
        return m.is_dir() and bool(list(m.glob("*.obj")) or list(m.glob("*.ply")))

    def ingest(zip_name):
        obj_id = zip_name[:-4]
        last_err = "unknown"
        for attempt in range(1, 5):
            try:
                zp = hf_hub_download(REPO, zip_name, repo_type="dataset", local_dir="/kaggle/working/gso_zips", token=os.environ.get("HF_TOKEN"))
                with tempfile.TemporaryDirectory() as td:
                    with zipfile.ZipFile(zp) as z:
                        z.extractall(td)
                    hits = list(Path(td).rglob("*.obj")) + list(Path(td).rglob("*.ply"))
                    if not hits:
                        raise RuntimeError("zip contains no .obj/.ply")
                    mdir = Path(GSO_DIR) / obj_id / "meshes"
                    mdir.mkdir(parents=True, exist_ok=True)
                    for h in hits:
                        shutil.copy(h, mdir / h.name)
                    for sib in hits[0].parent.iterdir():
                        if sib.suffix.lower() in (".mtl", ".png", ".jpg", ".jpeg"):
                            shutil.copy(sib, mdir / sib.name)
                    for tex in Path(td).rglob("*.png"):
                        if not (mdir / tex.name).exists():
                            shutil.copy(tex, mdir / tex.name)
                if _has_mesh(obj_id):
                    return True
                raise RuntimeError("mesh still missing after extract")
            except Exception as e:
                last_err = f"{type(e).__name__}: {str(e)[:90]}"
                print(f"  retry {attempt}/4 {obj_id} -> {last_err}", flush=True)
                _time.sleep(2.0 * attempt)
        print(f"  FAILED {obj_id} after 4 attempts -> {last_err}", flush=True)
        return False

    ok = 0
    for zname in wanted:
        if ingest(zname):
            ok += 1
        if ok >= len(wanted):
            break
    print(f"ingested {ok}/{len(wanted)} objects into {GSO_DIR}", flush=True)

    # Repair pass: only the objects that are still missing get retried (fresh downloads).
    still_missing = [n for n in TARGET_OBJECTS if not _has_mesh(n)]
    if still_missing:
        print(f"repair pass for {len(still_missing)} missing object(s): {still_missing}", flush=True)
        _time.sleep(5)
        for n in list(still_missing):
            if ingest(n + ".zip") and n in still_missing:
                still_missing.remove(n)
    if still_missing:
        print(f"WARNING: {len(still_missing)} GSO object(s) unavailable after retries: {still_missing}", flush=True)
        Path("/kaggle/working/outputs").mkdir(parents=True, exist_ok=True)
        Path("/kaggle/working/outputs/ingest_missing.json").write_text(json.dumps(still_missing))
        TARGET_OBJECTS = [n for n in TARGET_OBJECTS if n not in still_missing]
        NUM_SAMPLES = len(TARGET_OBJECTS)
        print(f"continuing with {NUM_SAMPLES} objects so the benchmark still completes", flush=True)
    shutil.rmtree("/kaggle/working/gso_zips", ignore_errors=True)

rendered = 0
for d in sorted(Path(GSO_DIR).iterdir()):
    if not d.is_dir():
        continue
    mcands = list((d/"meshes").glob("*.obj")) + list((d/"meshes").glob("*.ply"))
    if not mcands or (d/"images"/".upright_v6").exists():
        continue
    try:
        (d/"images").mkdir(parents=True, exist_ok=True)
        # Render 000.png (1024x1024 RGBA) and 001.png (256x256 RGB) directly from the native
        # 4096x4096 GSO UV texture map (no vertex-baking bottleneck or double-gamma washout).
        rgba_000 = render_gso_mesh_view(mcands[0], azimuth_deg=20.0, elevation_deg=15.0, resolution=1024, return_rgba=True)
        rgb_001 = render_gso_mesh_view(mcands[0], azimuth_deg=65.0, elevation_deg=15.0, resolution=256, return_rgba=False)
        if rgba_000 is None or rgb_001 is None:
            continue
        PILImage.fromarray(rgba_000, mode="RGBA").save(d/"images"/"000.png")
        PILImage.fromarray(rgb_001).save(d/"images"/"001.png")
        (d/"images"/".upright_v6").write_text("ok")
        rendered += 1
    except Exception as e:
        print("render failed:", d.name, str(e)[:100])

samples = load_gso_samples(GSO_DIR, num_samples=NUM_SAMPLES, object_ids=TARGET_OBJECTS)
_missing = [o for o in TARGET_OBJECTS if o not in {sm.sample_id for sm in samples}]
if _missing or len(samples) != len(TARGET_OBJECTS):
    raise RuntimeError(f"GSO pre-flight failed: {len(samples)}/{len(TARGET_OBJECTS)} ready, missing={_missing}")
print(f"GSO ready: rendered {rendered} upright RGBA views | loaded {len(samples)}/{len(TARGET_OBJECTS)} canonical samples ✅")

In [ ]:
# --- SPAR3D (Point-E 512-Pt Diffusion + Triplane DMTet + PBR, ~2.0 s/object on T4) ---
# Wrapped in top-level try/except so Papermill (allow_errors=False) never aborts the kernel!
import os, sys, gc, datetime, shutil, subprocess, types, traceback
from pathlib import Path
import packaging
import torch

try:
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
    os.environ["SPAR3D_LOW_VRAM"] = "0"
    os.environ["CC"] = "gcc"
    os.environ["CXX"] = "g++"
    os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
    os.environ["MAX_JOBS"] = "4"

    if "CODE_DIR" not in globals():
        CODE_DIR = "/kaggle/working/image-to-3d-benchmark"
    if str(CODE_DIR) not in sys.path:
        sys.path.insert(0, str(CODE_DIR))

    from src.engines.spar3d_engine import Spar3dEngine, _install_spar3d_compat_shims
    from src.evaluation.benchmark_runner import BenchmarkRunner

    _install_spar3d_compat_shims()

    # 1. Install Python dependencies FIRST (including ftfy, regex, loralib) BEFORE importing clip/alpha_clip!
    !pip install -q -c /tmp/constraints.txt einops jaxtyping omegaconf loralib ftfy regex timm kornia

    # 2. Clone CLIP & AlphaCLIP, patch pkg_resources, and add to sys.path
    for _repo_url, _dest in [
        ("https://github.com/openai/CLIP.git", "/tmp/CLIP"),
        ("https://github.com/SunzeY/AlphaCLIP.git", "/tmp/AlphaCLIP"),
    ]:
        if not os.path.isdir(os.path.join(_dest, ".git")):
            shutil.rmtree(_dest, ignore_errors=True)
            subprocess.run(["git", "clone", "--depth", "1", _repo_url, _dest], check=True)
        for _pyf in Path(_dest).rglob("*.py"):
            try:
                _t = _pyf.read_text()
                if "from pkg_resources import packaging" in _t:
                    _pyf.write_text(_t.replace("from pkg_resources import packaging", "import packaging"))
            except Exception:
                pass
        if _dest not in sys.path:
            sys.path.insert(0, _dest)

    import clip, alpha_clip
    print("[SPAR3D] clip and alpha_clip imported cleanly ✅")

    # 3. Clone Stability-AI/stable-point-aware-3d and build sm_75 CUDA extensions
    SPAR3D_DIR = "/tmp/stable-point-aware-3d"
    if not os.path.isdir(os.path.join(SPAR3D_DIR, ".git")):
        shutil.rmtree(SPAR3D_DIR, ignore_errors=True)
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/Stability-AI/stable-point-aware-3d.git", SPAR3D_DIR],
            check=True,
        )
    if SPAR3D_DIR not in sys.path:
        sys.path.insert(0, SPAR3D_DIR)

    try:
        import nvdiffrast.torch  # noqa: F401
    except ImportError:
        !pip install -q -c /tmp/constraints.txt --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git

    print("[SPAR3D] Building texture_baker and uv_unwrapper CUDA extensions (sm_75) …")
    _tb = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "--no-deps", f"{SPAR3D_DIR}/texture_baker"],
        capture_output=True, text=True,
    )
    _uw = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "--no-deps", f"{SPAR3D_DIR}/uv_unwrapper"],
        capture_output=True, text=True,
    )
    print(f"[SPAR3D] CUDA extensions: texture_baker={_tb.returncode}, uv_unwrapper={_uw.returncode}")

    sp_engine = Spar3dEngine(device="cuda", texture_resolution=1024, low_vram_mode=False)
    sp_engine.load_model()
    runner = BenchmarkRunner(
        dataset_path=GSO_DIR,
        engines=[sp_engine],
        num_samples=NUM_SAMPLES, object_ids=TARGET_OBJECTS,
        output_dir="/kaggle/working/outputs/spar3d",
        resume_json_path=None,
        save_mesh_limit=100,
        max_wall_minutes=None,
    )
    try:
        print(f"[SPAR3D] {datetime.datetime.now().strftime('%H:%M:%S')} Starting SPAR3D run...")
        runner.run()
        print(f"[SPAR3D] {datetime.datetime.now().strftime('%H:%M:%S')} SPAR3D COMPLETE ✅")
        runner.save_results("/kaggle/working/outputs/spar3d_results.json")
        display(runner.get_summary_df())
    finally:
        del runner, sp_engine
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
except Exception as e:
    print(f"[SPAR3D] Failed with {type(e).__name__}: {e}")
    traceback.print_exc()


In [ ]:
# --- TRELLIS (Sparse Structure Flow + SLAT Rectified Flow -> 256^3 FlexiCubes) ---
# Always runs TRELLIS. Cached v22 results were removed: they carry no code_sha,
# so importing them would silently reintroduce unverifiable numbers.
import os, sys, gc, glob, json, shutil, subprocess, datetime, types, traceback
from pathlib import Path
import torch

RUN_TRELLIS = True

try:
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
    os.environ["ATTN_BACKEND"] = "xformers"
    os.environ["SPCONV_ALGO"] = "native"
    os.environ["SPCONV_DISABLE_JIT"] = "1"
    os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
    os.environ["MAX_JOBS"] = "4"

    _trellis_out = Path("/kaggle/working/outputs/trellis_results.json")
    _trellis_out.parent.mkdir(parents=True, exist_ok=True)
    if not RUN_TRELLIS:
        raise RuntimeError(
            "RUN_TRELLIS=False is not allowed: cached v22 results carry no code_sha and must not "
            "enter a results table. Run TRELLIS instead."
        )
    else:
        !pip install -q -c /tmp/constraints.txt "pybind11==2.13.6" cumm-cu126 spconv-cu126
        _c8incs = sorted(glob.glob("/usr/local/lib/python3*/dist-packages/cumm/include")) + \
                  sorted(glob.glob("/opt/conda/lib/python3*/site-packages/cumm/include"))
        if _c8incs:
            os.environ["CPATH"] = _c8incs[0] + (os.pathsep + os.environ["CPATH"] if os.environ.get("CPATH") else "")

        try:
            import xformers  # noqa: F401
        except ImportError:
            os.system("pip install -q -c /tmp/constraints.txt --no-deps xformers")

        !pip install -q -c /tmp/constraints.txt pytorch-lightning omegaconf easydict "imageio[ffmpeg]" timm kornia open3d
        try:
            import nvdiffrast.torch  # noqa: F401
        except ImportError:
            !pip install -q -c /tmp/constraints.txt --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git

        def _ensure_checkout(repo_url, dest, pin):
            if os.path.isdir(dest) and not os.path.isdir(os.path.join(dest, ".git")):
                shutil.rmtree(dest, ignore_errors=True)
            if not os.path.isdir(os.path.join(dest, ".git")):
                subprocess.run(["git", "clone", "--depth", "1", "--recurse-submodules", repo_url, dest], check=True)
            subprocess.run(["git", "-C", dest, "checkout", "-f", pin], capture_output=True)

        _ensure_checkout("https://github.com/microsoft/TRELLIS.git", "/tmp/TRELLIS", "442aa1e1afb9014e80681d3bf604e8d728a86ee7")
        if "/tmp/TRELLIS" not in sys.path:
            sys.path.insert(0, "/tmp/TRELLIS")

        # Patch TRELLIS base.py: surface swallowed primary-load errors instead of
        # silently falling back to a bogus HF repo id (bare `v` -> 401).
        _base_py = "/tmp/TRELLIS/trellis/pipelines/base.py"
        _base_src = open(_base_py).read()
        _old = "            except:\n                _models[k] = models.from_pretrained(v)"
        _new = (
            "            except Exception as _primary_err:\n"
            "                print(f\"[TRELLIS-loader] {k}: primary load failed: {type(_primary_err).__name__}: {_primary_err}\")\n"
            "                if os.path.isdir(path):\n"
            "                    raise\n"
            "                _models[k] = models.from_pretrained(v)"
        )
        if _old in _base_src:
            open(_base_py, "w").write(_base_src.replace(_old, _new))
            print("[TRELLIS] Patched base.py to surface primary-load errors ✅")
        elif "[TRELLIS-loader]" in _base_src:
            print("[TRELLIS] base.py already patched ✅")
        else:
            print("[TRELLIS] WARNING: base.py patch anchor not found (upstream changed?)")

        if "kaolin" not in sys.modules:
            _kaolin = types.ModuleType("kaolin")
            _kaolin_utils = types.ModuleType("kaolin.utils")
            _kaolin_testing = types.ModuleType("kaolin.utils.testing")
            _kaolin_testing.check_tensor = lambda tensor, shape=None, throw=True, **_kw: True
            _kaolin.utils = _kaolin_utils
            _kaolin.utils.testing = _kaolin_testing
            sys.modules["kaolin"] = _kaolin
            sys.modules["kaolin.utils"] = _kaolin_utils
            sys.modules["kaolin.utils.testing"] = _kaolin_testing

        _ensure_checkout("https://github.com/JeffreyXiang/diffoctreerast.git", "/tmp/extensions/diffoctreerast", "b09c20b84ec3aace4729e6e18a613112320eca3a")
        !pip install -q --no-build-isolation /tmp/extensions/diffoctreerast
        !pip install -q git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8

        from src.engines.trellis_engine import TrellisEngine
        from src.evaluation.benchmark_runner import BenchmarkRunner

        tr_engine2 = TrellisEngine(device="cuda", output_format="mesh", slat_steps=8, sparse_steps=8, sparse_cfg_strength=7.5, slat_cfg_strength=3.0, staged_offload=True)
        tr_engine2.load_model()
        # v29 TRELLIS rows come from the gamma-darkened runner, so all objects are re-rendered.
        runner = BenchmarkRunner(dataset_path=GSO_DIR, engines=[tr_engine2], num_samples=NUM_SAMPLES, object_ids=TARGET_OBJECTS, output_dir="/kaggle/working/outputs/trellis", resume_json_path=None, save_mesh_limit=100, max_wall_minutes=None)
        try:
            runner.run()
            runner.save_results("/kaggle/working/outputs/trellis_results.json")
            display(runner.get_summary_df())
        finally:
            del runner, tr_engine2
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
except Exception as e:
    print(f"[TRELLIS] Error: {type(e).__name__}: {e}")
    traceback.print_exc()


In [ ]:
import glob, json
import pandas as pd
import matplotlib.pyplot as plt

rows = []
for p in sorted(glob.glob("/kaggle/working/outputs/*_results.json")):
    try:
        rows += json.load(open(p))
    except Exception as e:
        print("skip", p, str(e)[:80])

df = pd.DataFrame(rows)
_before = len(df)
df = df.drop_duplicates(subset=["engine", "sample_id"], keep="last")
if len(df) != _before:
    print(f"\u26a0 dropped {_before - len(df)} duplicate (engine, sample_id) rows")
_counts = df.groupby("engine").size().to_dict()
_bad = {e: int(c) for e, c in _counts.items() if c != NUM_SAMPLES}
if _bad:
    print(f"\u26a0 row-count mismatch vs NUM_SAMPLES={NUM_SAMPLES}: {_bad} — results incomplete!")
df.to_csv("/kaggle/working/outputs/unified_results.csv", index=False)

num_cols = ["latency_s", "vram_peak_gb", "chamfer_distance", "f_score_0.1", "f_score_0.2", "f_score_0.5", "psnr", "lpips"]
cols = [c for c in num_cols if c in df.columns]
if cols and not df.empty:
    summary = df.groupby("engine")[cols].agg(["mean", "std"]).round(4)
    print("=== UNIFIED BENCHMARK RESULTS (mean +/- std) ===")
    print(summary.to_string())
    with open("/kaggle/working/outputs/unified_table.tex", "w") as f:
        f.write(summary.reset_index().to_latex(index=False, na_rep="--"))

    colors = {"TripoSR": "#1f77b4", "LGM": "#ff7f0e", "TRELLIS": "#2ca02c", "SPAR3D": "#9467bd"}
    means = df.groupby("engine")[cols].mean()
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for eng, row in means.iterrows():
        c = colors.get(eng, "grey")
        if "f_score_0.1" in row and not pd.isna(row.get("f_score_0.1")):
            axes[0].scatter(row["latency_s"], row["f_score_0.1"], s=180, c=c, label=eng, zorder=5)
        if "chamfer_distance" in row and not pd.isna(row.get("chamfer_distance")):
            axes[1].scatter(row["latency_s"], row["chamfer_distance"], s=180, c=c, label=eng, zorder=5)
    axes[0].set_xlabel("Latency (s)"); axes[0].set_ylabel("F-Score @ 0.1"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].set_xlabel("Latency (s)"); axes[1].set_ylabel("Chamfer Distance"); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("/kaggle/working/outputs/pareto_frontier.png", dpi=150)
    print("saved: unified_results.csv, unified_table.tex, pareto_frontier.png ✅")

In [ ]:
import shutil, os
from pathlib import Path

WORK = Path("/kaggle/working")
KEEP = {"outputs"}
if WORK.exists():
    for item in sorted(WORK.iterdir()):
        if item.name in KEEP:
            continue
        try:
            if item.is_dir():
                shutil.rmtree(item)
            else:
                item.unlink()
        except Exception:
            pass
    print("saved outputs:", sorted(p.name for p in (WORK / "outputs").iterdir()))